In [ ]:
from helper_client_setup import client, model

This method is not used

In [ ]:
import requests

def get_weather(location):
    if not location or location.strip() == "":
        raise ValueError("Location cannot be empty")

    geourl = f"https://geocoding-api.open-meteo.com/v1/search?name={location}&count=1&format=json"
    params = {
        "appid": "1234567890",
        "q": location,
        "units": "metric"
    }
    response = requests.get(geourl, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()
    return data



In [ ]:
from datetime import datetime
from anthropic.types import ToolParam

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})


In [ ]:
message = []

message.append(
    {
        "role": "user",
        "content": "What is the exact time formated as HH:MM:SS? what is your name?"
    }
)

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=message,
    tools=[get_current_datetime_schema],
    tool_choice={"type": "auto"}
)

message.append({
    "role": "assistant",
    "content": response.content
})

message


In [ ]:
# Execute the tool
tool_use = response.content[0]
result = get_current_datetime(**tool_use.input)
result

In [ ]:
message.append({
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": tool_use.id,
            "content": result,
            "is_error": False
        }
    ]
})

message

In [ ]:
final_response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=message,
    tools=[get_current_datetime_schema]
)
print(final_response)
print("--------------------------------")
print(final_response.content[0].text)